# FoxCode 研究实验：Context / Memory / Skill

假模型 + 真实临时文件工具。产生失败→恢复经验，再用独立试用任务验证候选技能。


In [ ]:
import tempfile
from pathlib import Path
from fox_ai.src import AssistantMessage, ToolCall, Done
from fox_coding_agent.src import CodingAgent, Config

async def fake_model(model, context, options):
    last = context.messages[-1]
    if last.role == "user":
        if last.content.startswith("Verify"):
            call = ToolCall("r", "Read", {"file_path": "demo.py"})
        else:
            call = ToolCall("w", "Write", {"file_path": "demo.py", "content": "value = 41"})
    elif last.name == "Write":
        call = ToolCall("failed", "Edit", {"file_path": "demo.py", "old_text": "value = 40", "new_text": "value = 42"})
    elif last.name == "Edit" and last.is_error:
        call = ToolCall("fixed", "Edit", {"file_path": "demo.py", "old_text": "value = 41", "new_text": "value = 42"})
    else:
        yield Done(AssistantMessage("Done"))
        return
    yield Done(AssistantMessage(tool_calls=[call]))

async def research_demo():
    with tempfile.TemporaryDirectory() as directory:
        coding = CodingAgent(Config(cwd=Path(directory), model="mock"), stream_fn=fake_model)
        try:
            coding.memory.add("demo parser uses pytest", memory_type="semantic", scope=coding.scope)
            events = [e async for e in coding.run("Fix demo parser; 不要增加依赖")]
            assert events[-1].data["status"] == "completed"
            assert coding.memory_hits
            candidate = coding.skills.all()[0]
            assert candidate.status == "candidate"
            for index in range(2):
                events = [e async for e in coding.run(f"Verify demo parser {index}", trial_skill=candidate.name)]
                # usefulness 判分来自本例显式文件断言，不来自模型自称 Done。
                assert (Path(directory) / "demo.py").read_text() == "value = 42"
                coding.validate_skill(candidate.name, coding.trajectory["id"], useful=True)
            assert coding.skills.get(candidate.name).status == "active"
            print(coding.snapshot())
        finally:
            coding.close()

await research_demo()


Context 是当前输入，Memory 保存跨任务经验，Skill 是可复用策略。压缩与检索实现独立于 core。进一步改变 context_budget、删去 memory/skill 注入或修改 Utility 权重做消融；完整 trajectory 在临时项目 .foxcode/research/ 中，实验结束清理。
